In [2]:
!python -V
!pip install catboost lightgbm optuna xgboost --quiet
!pip list | grep -E 'catboost|lightgbm|optuna|xgboost|pandas|numpy|scikit'

Python 3.12.13
catboost                                 1.2.10
geopandas                                1.1.3
lightgbm                                 4.6.0
numpy                                    2.4.6
optuna                                   4.8.0
pandas                                   2.3.3
pandas-datareader                        0.10.0
pandas-gbq                               0.30.0
pandas-profiling                         3.6.6
pandas-stubs                             2.2.2.240909
pandasql                                 0.7.3
scikit-image                             0.25.2
scikit-learn                             1.6.1
scikit-multilearn                        0.2.0
scikit-optimize                          0.10.2
scikit-plot                              0.3.7
scikit-surprise                          1.1.4
sklearn-pandas                           2.2.0
xgboost                                  3.2.0


In [3]:
!pip install numpy
import numpy as np
import pandas as pd
import re
import warnings
import optuna
import random
from scipy.optimize import minimize
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostRegressor, Pool
import lightgbm as lgb
import xgboost as xgb

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

In [4]:
df_raw = pd.read_csv('/kaggle/input/datasets/redac6767/ml4454/train_2.csv')
print('Raw shape:', df_raw.shape)
print('Columns:', df_raw.columns.tolist())

Raw shape: (485082, 24)
Columns: ['new_id', 'Год', 'Месяц', 'Среднее количество промо товаров в чеке', 'Среднее количество товаров в чеке', 'Среднее количество отмен', 'Рабочие часы в день', 'Дата открытия, категориальный', 'Торговая площадь, категориальный', 'Населенный пункт', 'Регион', 'Численность населения', 'Количество домохозяйств', 'Трафик пеший, в час', 'Трафик авто, в час', 'Маркетплейсы, доставки, постаматы (100 м)', 'Медицинские уч. и аптеки (300 м)', 'Школы (300 м)', 'Остановки (300 м)', 'Продуктовые магазины (500 м)', 'Пятерочки (500 м)', 'Количество касс', 'Флаг алкогольной лицензии', 'РТО']


In [5]:
COL_RENAME = {
    'Год':                                         'year_col',
    'Месяц':                                       'month_col',
    'РТО':                                         'rto',
    'Среднее количество промо товаров в чеке':     'avg_promo_items',
    'Среднее количество товаров в чеке':           'avg_items_in_check',
    'Среднее количество отмен':                    'avg_cancellations',
    'Рабочие часы в день':                         'working_hours',
    'Дата открытия, категориальный':               'open_date_cat',
    'Торговая площадь, категориальный':            'floor_area_cat',
    'Населенный пункт':                            'city',
    'Регион':                                      'region',
    'Численность населения':                       'population',
    'Количество домохозяйств':                     'households',
    'Трафик пеший, в час':                         'foot_traffic',
    'Трафик авто, в час':                          'car_traffic',
    'Маркетплейсы, доставки, постаматы (100 м)':  'marketplaces_100m',
    'Медицинские уч. и аптеки (300 м)':           'medical_300m',
    'Школы (300 м)':                              'schools_300m',
    'Остановки (300 м)':                          'stops_300m',
    'Продуктовые магазины (500 м)':               'grocery_500m',
    'Пятерочки (500 м)':                          'pyaterochka_500m',
    'Количество касс':                            'num_cashiers',
    'Флаг алкогольной лицензии':                  'alcohol_license',
}

df = df_raw.rename(columns=COL_RENAME)

df['month'] = pd.to_datetime(
    df['year_col'].astype(str) + '-' + df['month_col'].astype(str).str.zfill(2) + '-01'
)
df = df.drop(columns=['year_col', 'month_col'])
df = df.sort_values(['new_id', 'month']).reset_index(drop=True)
df['rto'] = df['rto'].clip(lower=1)

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Month range:', df['month'].min(), '-', df['month'].max())
print('Unique stores:', df['new_id'].nunique())

Shape: (485082, 23)
Columns: ['new_id', 'avg_promo_items', 'avg_items_in_check', 'avg_cancellations', 'working_hours', 'open_date_cat', 'floor_area_cat', 'city', 'region', 'population', 'households', 'foot_traffic', 'car_traffic', 'marketplaces_100m', 'medical_300m', 'schools_300m', 'stops_300m', 'grocery_500m', 'pyaterochka_500m', 'num_cashiers', 'alcohol_license', 'rto', 'month']
Month range: 2023-01-01 00:00:00 - 2025-02-01 00:00:00
Unique stores: 18657


In [6]:
print('=== EDA ===')
print('Missing values:')
print(df.isnull().sum())
print()
print('RTO stats:')
print(df['rto'].describe())
print()
print('Categorical columns:')
for c in ['open_date_cat', 'floor_area_cat', 'city', 'region', 'alcohol_license']:
    print(f'  {c}: {df[c].nunique()} unique | values: {df[c].unique()[:5]}')

=== EDA ===
Missing values:
new_id                0
avg_promo_items       0
avg_items_in_check    0
avg_cancellations     0
working_hours         0
open_date_cat         0
floor_area_cat        0
city                  0
region                0
population            0
households            0
foot_traffic          0
car_traffic           0
marketplaces_100m     0
medical_300m          0
schools_300m          0
stops_300m            0
grocery_500m          0
pyaterochka_500m      0
num_cashiers          0
alcohol_license       0
rto                   0
month                 0
dtype: int64

RTO stats:
count    4.850820e+05
mean     8.853480e+07
std      4.824900e+07
min      1.179116e+06
25%      5.627279e+07
50%      7.521094e+07
75%      1.057677e+08
max      6.557204e+08
Name: rto, dtype: float64

Categorical columns:
  open_date_cat: 3 unique | values: ['Новый' 'Средний по возрасту' 'Открыт давно']
  floor_area_cat: 4 unique | values: ['Большой' 'Средний' 'Маленький' 'Очень большой']
 

In [7]:
CAT_COLS = ['open_date_cat', 'floor_area_cat', 'city', 'region', 'alcohol_license']

NUM_STATIC_COLS = [
    'avg_promo_items', 'avg_items_in_check', 'avg_cancellations',
    'working_hours', 'population', 'households',
    'foot_traffic', 'car_traffic',
    'marketplaces_100m', 'medical_300m', 'schools_300m',
    'stops_300m', 'grocery_500m', 'pyaterochka_500m',
    'num_cashiers'
]

ID_COL = 'new_id'

print('Cat cols:', CAT_COLS)
print('Num static cols:', NUM_STATIC_COLS)

Cat cols: ['open_date_cat', 'floor_area_cat', 'city', 'region', 'alcohol_license']
Num static cols: ['avg_promo_items', 'avg_items_in_check', 'avg_cancellations', 'working_hours', 'population', 'households', 'foot_traffic', 'car_traffic', 'marketplaces_100m', 'medical_300m', 'schools_300m', 'stops_300m', 'grocery_500m', 'pyaterochka_500m', 'num_cashiers']


In [8]:
def mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask   = (y_true > 0) & ~np.isnan(y_true) & ~np.isnan(y_pred)
    return 100.0 * np.mean(np.abs((y_pred[mask] - y_true[mask]) / y_true[mask]))

def score_to_points(m):
    return round(100 * ((100 - min(m, 100)) / 100) ** 2, 2)

for v in [14, 12, 10, 8, 6, 5]:
    print(f'MAPE {v}% -> {score_to_points(v)} pts')

MAPE 14% -> 73.96 pts
MAPE 12% -> 77.44 pts
MAPE 10% -> 81.0 pts
MAPE 8% -> 84.64 pts
MAPE 6% -> 88.36 pts
MAPE 5% -> 90.25 pts


In [9]:
sorted_months = sorted(df['month'].unique())
val_month_cv  = sorted_months[-1]
print('All months:', [str(m)[:7] for m in sorted_months])
print('Val month:', str(val_month_cv)[:7])

All months: ['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02']
Val month: 2025-02


In [10]:
def build_features(df_in):
    d   = df_in.copy().sort_values([ID_COL, 'month'])
    grp = d.groupby(ID_COL)['rto']

    for lag in [1, 2, 3, 4, 5, 6, 9, 12]:
        d[f'lag_{lag}'] = grp.shift(lag)

    for w in [2, 3, 6, 9, 12]:
        d[f'roll_mean_{w}'] = grp.transform(lambda x, w=w: x.shift(1).rolling(w, min_periods=1).mean())
        d[f'roll_std_{w}']  = grp.transform(lambda x, w=w: x.shift(1).rolling(w, min_periods=1).std())
        d[f'roll_min_{w}']  = grp.transform(lambda x, w=w: x.shift(1).rolling(w, min_periods=1).min())
        d[f'roll_max_{w}']  = grp.transform(lambda x, w=w: x.shift(1).rolling(w, min_periods=1).max())
        d[f'roll_med_{w}']  = grp.transform(lambda x, w=w: x.shift(1).rolling(w, min_periods=1).median())

    d['ratio_1_2']   = d['lag_1'] / d['lag_2'].replace(0, np.nan)
    d['ratio_2_3']   = d['lag_2'] / d['lag_3'].replace(0, np.nan)
    d['ratio_1_3']   = d['lag_1'] / d['lag_3'].replace(0, np.nan)
    d['ratio_1_6']   = d['lag_1'] / d['lag_6'].replace(0, np.nan)
    d['ratio_1_12']  = d['lag_1'] / d['lag_12'].replace(0, np.nan)
    d['ratio_3_12']  = d['lag_3'] / d['lag_12'].replace(0, np.nan)
    d['diff_1_2']    = d['lag_1'] - d['lag_2']
    d['diff_1_3']    = d['lag_1'] - d['lag_3']
    d['diff_1_12']   = d['lag_1'] - d['lag_12']

    rm3  = d['roll_mean_3'].replace(0, np.nan)
    rm6  = d['roll_mean_6'].replace(0, np.nan)
    rm12 = d['roll_mean_12'].replace(0, np.nan)
    d['cv_3']        = d['roll_std_3']  / rm3
    d['cv_6']        = d['roll_std_6']  / rm6
    d['trend_3_6']   = d['roll_mean_3'] / rm6
    d['trend_6_12']  = d['roll_mean_6'] / rm12
    d['trend_3_12']  = d['roll_mean_3'] / rm12
    d['range_6']     = d['roll_max_6']  - d['roll_min_6']
    d['range_12']    = d['roll_max_12'] - d['roll_min_12']

    d['log_lag_1']   = np.log1p(d['lag_1'])
    d['log_lag_3']   = np.log1p(d['lag_3'])
    d['log_lag_12']  = np.log1p(d['lag_12'])

    d['month_num']   = d['month'].dt.month
    d['year']        = d['month'].dt.year
    d['month_sin']   = np.sin(2 * np.pi * d['month_num'] / 12)
    d['month_cos']   = np.cos(2 * np.pi * d['month_num'] / 12)
    d['quarter']     = d['month'].dt.quarter
    d['is_q1']       = (d['quarter'] == 1).astype(int)
    d['is_march']    = (d['month_num'] == 3).astype(int)

    gm_med           = d.groupby('month_num')['rto'].transform('median')
    d['seasonal_idx']= d['rto'] / gm_med.replace(0, np.nan)

    sm               = d.groupby(ID_COL)['rto'].transform('median')
    d['store_size']      = sm
    d['log_store_size']  = np.log1p(sm)

    d['promo_x_traffic'] = d['avg_promo_items'] * d['foot_traffic']
    d['items_x_hours']   = d['avg_items_in_check'] * d['working_hours']
    d['competition_idx'] = d['grocery_500m'] + d['pyaterochka_500m']
    d['infra_idx']       = d['stops_300m'] + d['schools_300m'] + d['medical_300m']
    d['total_traffic']   = d['foot_traffic'] + d['car_traffic']
    d['households_per_cash'] = d['households'] / d['num_cashiers'].replace(0, np.nan)
    d['pop_per_cash']    = d['population'] / d['num_cashiers'].replace(0, np.nan)

    d['target_ratio']     = grp.transform(lambda x: x / x.shift(1))
    d['log_target_ratio'] = np.log(d['target_ratio'].clip(0.01, 100))

    return d

print('Building features...')
df_feat = build_features(df)
print('Features shape:', df_feat.shape)

Building features...
Features shape: (485082, 94)


In [11]:
RATIO_LOW  = df_feat['target_ratio'].quantile(0.005)
RATIO_HIGH = df_feat['target_ratio'].quantile(0.995)
LOG_LOW    = np.log(max(RATIO_LOW, 1e-6))
LOG_HIGH   = np.log(max(RATIO_HIGH, 1e-6))
print(f'Ratio clip: [{RATIO_LOW:.4f}, {RATIO_HIGH:.4f}]')
print(f'Log-ratio clip: [{LOG_LOW:.4f}, {LOG_HIGH:.4f}]')

Ratio clip: [0.7151, 1.4096]
Log-ratio clip: [-0.3354, 0.3433]


In [12]:
LAG_FEATS   = [f'lag_{i}' for i in [1, 2, 3, 4, 5, 6, 9, 12]]
ROLL_FEATS  = [f'roll_{s}_{w}' for s in ['mean','std','min','max','med'] for w in [2, 3, 6, 9, 12]]
RATIO_FEATS = ['ratio_1_2','ratio_2_3','ratio_1_3','ratio_1_6','ratio_1_12','ratio_3_12',
               'diff_1_2','diff_1_3','diff_1_12',
               'cv_3','cv_6','trend_3_6','trend_6_12','trend_3_12','range_6','range_12']
LOG_FEATS   = ['log_lag_1','log_lag_3','log_lag_12']
CAL_FEATS   = ['month_num','year','month_sin','month_cos','quarter','is_q1','is_march']
STORE_FEATS = ['store_size','log_store_size']
INTER_FEATS = ['promo_x_traffic','items_x_hours','competition_idx',
               'infra_idx','total_traffic','households_per_cash','pop_per_cash']

ALL_NUM_FEATS = LAG_FEATS + ROLL_FEATS + RATIO_FEATS + LOG_FEATS + CAL_FEATS + STORE_FEATS + INTER_FEATS + NUM_STATIC_COLS

assert len(ALL_NUM_FEATS) == len(set(ALL_NUM_FEATS)), 'Duplicate feature names in ALL_NUM_FEATS!'

print('Num features:', len(ALL_NUM_FEATS))
print('Cat features:', CAT_COLS)

Num features: 83
Cat features: ['open_date_cat', 'floor_area_cat', 'city', 'region', 'alcohol_license']


In [13]:
le_dict = {}
for c in CAT_COLS:
    le = LabelEncoder()
    df_feat[c + '_enc'] = le.fit_transform(df_feat[c].astype(str).fillna('NA'))
    le_dict[c] = le

CAT_ENC_COLS  = [c + '_enc' for c in CAT_COLS]
ALL_ENC_FEATS = ALL_NUM_FEATS + CAT_ENC_COLS

assert len(ALL_ENC_FEATS) == len(set(ALL_ENC_FEATS)), 'Duplicate names in ALL_ENC_FEATS!'
print('Total encoded features:', len(ALL_ENC_FEATS))
print('Enc cat cols:', CAT_ENC_COLS)

Total encoded features: 88
Enc cat cols: ['open_date_cat_enc', 'floor_area_cat_enc', 'city_enc', 'region_enc', 'alcohol_license_enc']


In [14]:
prev_month_cv = sorted_months[sorted_months.index(val_month_cv) - 1]

train_mask = (
    (df_feat['month'] <= prev_month_cv) &
    df_feat['target_ratio'].notna() &
    df_feat['lag_1'].notna()
)
train_df = df_feat[train_mask].copy()
val_df   = df_feat[df_feat['month'] == val_month_cv].copy()

train_df['target_ratio']     = train_df['target_ratio'].clip(RATIO_LOW, RATIO_HIGH)
train_df['log_target_ratio'] = np.log(train_df['target_ratio'])

X_tr_enc = train_df[ALL_ENC_FEATS].copy()
y_tr_ratio = train_df['target_ratio'].copy()
y_tr_log   = train_df['log_target_ratio'].copy()

X_vl_enc = val_df[ALL_ENC_FEATS].copy()
y_val_true = val_df['rto'].values
y_val_lag1 = val_df['lag_1'].values
val_true_ratio = val_df['target_ratio'].clip(RATIO_LOW, RATIO_HIGH).fillna(1.0).values
val_true_log   = np.log(np.clip(val_true_ratio, 0.01, 100))

ALL_CB_FEATS = ALL_NUM_FEATS + CAT_COLS
X_tr_cat = train_df[ALL_CB_FEATS].copy()
X_vl_cat = val_df[ALL_CB_FEATS].copy()
for c in CAT_COLS:
    X_tr_cat[c] = X_tr_cat[c].astype(str).fillna('NA')
    X_vl_cat[c] = X_vl_cat[c].astype(str).fillna('NA')

print('Train:', X_tr_enc.shape, ' Val:', X_vl_enc.shape)

Train: (447768, 88)  Val: (18657, 88)


In [15]:
try:
    import subprocess
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    USE_GPU = result.returncode == 0
except Exception:
    USE_GPU = False

CB_TASK  = 'GPU' if USE_GPU else 'CPU'
CB_DEPTH = 10    if USE_GPU else 8
CB_ITERS = 10000 if USE_GPU else 5000
print(f'Device: {CB_TASK} | depth={CB_DEPTH} | iters={CB_ITERS}')

Device: CPU | depth=8 | iters=5000


In [16]:
pool_tr_cat = Pool(X_tr_cat, y_tr_ratio, cat_features=CAT_COLS)
pool_vl_cat = Pool(X_vl_cat, val_true_ratio, cat_features=CAT_COLS)

cb_model = CatBoostRegressor(
    iterations=CB_ITERS,
    depth=CB_DEPTH,
    learning_rate=0.01,
    l2_leaf_reg=3,
    min_data_in_leaf=10,
    subsample=0.8,
    colsample_bylevel=0.8,
    loss_function='MAPE',
    eval_metric='MAPE',
    random_seed=SEED,
    early_stopping_rounds=300,
    task_type=CB_TASK,
    verbose=500,
    thread_count=-1
)
cb_model.fit(pool_tr_cat, eval_set=pool_vl_cat, use_best_model=True)
cb_best_iters = cb_model.best_iteration_
print('CatBoost best iter:', cb_best_iters)

pred_ratio_cb = np.clip(cb_model.predict(Pool(X_vl_cat, cat_features=CAT_COLS)), RATIO_LOW, RATIO_HIGH)
pred_cb_val   = np.clip(y_val_lag1 * pred_ratio_cb, 1, None)
print(f'CatBoost MAPE: {mape(y_val_true, pred_cb_val):.4f}% -> {score_to_points(mape(y_val_true, pred_cb_val))} pts')

0:	learn: 0.0727491	test: 0.0475642	best: 0.0475642 (0)	total: 554ms	remaining: 46m 9s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.04756419161
bestIteration = 0

Shrink model to first 1 iterations.
CatBoost best iter: 0
CatBoost MAPE: 5.7090% -> 88.91 pts


In [17]:
def lgb_objective(trial):
    params = {
        'objective':         'mape',
        'metric':            'mape',
        'n_estimators':      CB_ITERS,
        'learning_rate':     trial.suggest_float('lr', 0.005, 0.05, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 63, 511),
        'max_depth':         trial.suggest_int('max_depth', 5, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 1.0),
        'random_state':      SEED,
        'n_jobs':           -1,
        'verbose':          -1,
    }
    m = lgb.LGBMRegressor(**params)
    m.fit(
        X_tr_enc, y_tr_ratio,
        eval_set=[(X_vl_enc, val_true_ratio)],
        callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(-1)]
    )
    p = np.clip(y_val_lag1 * np.clip(m.predict(X_vl_enc), RATIO_LOW, RATIO_HIGH), 1, None)
    return mape(y_val_true, p)

study_lgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb.optimize(lgb_objective, n_trials=60, show_progress_bar=True)
print('LGB best MAPE:', round(study_lgb.best_value, 4), '%')
best_lgb_params = study_lgb.best_params

lgb_model = lgb.LGBMRegressor(
    objective='mape', metric='mape',
    n_estimators=CB_ITERS,
    learning_rate=best_lgb_params['lr'],
    num_leaves=best_lgb_params['num_leaves'],
    max_depth=best_lgb_params['max_depth'],
    min_child_samples=best_lgb_params['min_child_samples'],
    subsample=best_lgb_params['subsample'],
    colsample_bytree=best_lgb_params['colsample_bytree'],
    reg_alpha=best_lgb_params['reg_alpha'],
    reg_lambda=best_lgb_params['reg_lambda'],
    min_split_gain=best_lgb_params['min_split_gain'],
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_model.fit(
    X_tr_enc, y_tr_ratio,
    eval_set=[(X_vl_enc, val_true_ratio)],
    callbacks=[lgb.early_stopping(300, verbose=False), lgb.log_evaluation(500)]
)
lgb_best_iters = lgb_model.best_iteration_
print('LGB best iter:', lgb_best_iters)

pred_ratio_lgb = np.clip(lgb_model.predict(X_vl_enc), RATIO_LOW, RATIO_HIGH)
pred_lgb_val   = np.clip(y_val_lag1 * pred_ratio_lgb, 1, None)
print(f'LightGBM MAPE: {mape(y_val_true, pred_lgb_val):.4f}% -> {score_to_points(mape(y_val_true, pred_lgb_val))} pts')

  0%|          | 0/60 [00:00<?, ?it/s]

LGB best MAPE: 5.6146 %
[500]	valid_0's mape: 0.0431668
[1000]	valid_0's mape: 0.0364572
[1500]	valid_0's mape: 0.0344048
[2000]	valid_0's mape: 0.034087
LGB best iter: 2173
LightGBM MAPE: 4.2403% -> 91.7 pts


In [19]:
def xgb_objective(trial):
    params = {
        'objective':        'reg:absoluteerror',
        'n_estimators':     CB_ITERS,
        'learning_rate':    trial.suggest_float('lr', 0.005, 0.05, log=True),
        'max_depth':        trial.suggest_int('max_depth', 4, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 30),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
        'random_state':     SEED,
        'n_jobs':          -1,
        'tree_method':     'hist',
        'early_stopping_rounds': 150,
    }
    m = xgb.XGBRegressor(**params)
    m.fit(
        X_tr_enc, y_tr_ratio,
        eval_set=[(X_vl_enc, val_true_ratio)],
        verbose=False
    )
    p = np.clip(y_val_lag1 * np.clip(m.predict(X_vl_enc), RATIO_LOW, RATIO_HIGH), 1, None)
    return mape(y_val_true, p)

study_xgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(xgb_objective, n_trials=60, show_progress_bar=True)
print('XGB best MAPE:', round(study_xgb.best_value, 4), '%')
best_xgb_params = study_xgb.best_params

xgb_model = xgb.XGBRegressor(
    objective='reg:absoluteerror',
    n_estimators=CB_ITERS,
    learning_rate=best_xgb_params['lr'],
    max_depth=best_xgb_params['max_depth'],
    min_child_weight=best_xgb_params['min_child_weight'],
    subsample=best_xgb_params['subsample'],
    colsample_bytree=best_xgb_params['colsample_bytree'],
    reg_alpha=best_xgb_params['reg_alpha'],
    reg_lambda=best_xgb_params['reg_lambda'],
    gamma=best_xgb_params['gamma'],
    early_stopping_rounds=300,
    random_state=SEED, n_jobs=-1, tree_method='hist'
)
xgb_model.fit(
    X_tr_enc, y_tr_ratio,
    eval_set=[(X_vl_enc, val_true_ratio)],
    verbose=500
)
xgb_best_iters = xgb_model.best_iteration
print('XGB best iter:', xgb_best_iters)

pred_ratio_xgb = np.clip(xgb_model.predict(X_vl_enc), RATIO_LOW, RATIO_HIGH)
pred_xgb_val   = np.clip(y_val_lag1 * pred_ratio_xgb, 1, None)
print(f'XGBoost MAPE: {mape(y_val_true, pred_xgb_val):.4f}% -> {score_to_points(mape(y_val_true, pred_xgb_val))} pts')

  0%|          | 0/60 [00:00<?, ?it/s]

XGB best MAPE: 4.4905 %
[0]	validation_0-mae:0.05136
[500]	validation_0-mae:0.04084
[1000]	validation_0-mae:0.03823
[1500]	validation_0-mae:0.03751
[2000]	validation_0-mae:0.03734
[2500]	validation_0-mae:0.03704
[3000]	validation_0-mae:0.03691
[3500]	validation_0-mae:0.03677
[3779]	validation_0-mae:0.03677
XGB best iter: 3479
XGBoost MAPE: 4.4905% -> 91.22 pts


In [20]:
def lgb_log_objective(trial):
    params = {
        'objective':         'regression_l1',
        'metric':            'mae',
        'n_estimators':      CB_ITERS,
        'learning_rate':     trial.suggest_float('lr', 0.005, 0.05, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 63, 511),
        'max_depth':         trial.suggest_int('max_depth', 5, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state':      SEED,
        'n_jobs':           -1,
        'verbose':          -1,
    }
    m = lgb.LGBMRegressor(**params)
    m.fit(
        X_tr_enc, y_tr_log,
        eval_set=[(X_vl_enc, val_true_log)],
        callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(-1)]
    )
    p_ratio = np.clip(np.exp(np.clip(m.predict(X_vl_enc), LOG_LOW, LOG_HIGH)), RATIO_LOW, RATIO_HIGH)
    p = np.clip(y_val_lag1 * p_ratio, 1, None)
    return mape(y_val_true, p)

study_lgb_log = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb_log.optimize(lgb_log_objective, n_trials=40, show_progress_bar=True)
print('LGB-log best MAPE:', round(study_lgb_log.best_value, 4), '%')
best_lgb_log_params = study_lgb_log.best_params

lgb_log_model = lgb.LGBMRegressor(
    objective='regression_l1', metric='mae',
    n_estimators=CB_ITERS,
    learning_rate=best_lgb_log_params['lr'],
    num_leaves=best_lgb_log_params['num_leaves'],
    max_depth=best_lgb_log_params['max_depth'],
    min_child_samples=best_lgb_log_params['min_child_samples'],
    subsample=best_lgb_log_params['subsample'],
    colsample_bytree=best_lgb_log_params['colsample_bytree'],
    reg_alpha=best_lgb_log_params['reg_alpha'],
    reg_lambda=best_lgb_log_params['reg_lambda'],
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_log_model.fit(
    X_tr_enc, y_tr_log,
    eval_set=[(X_vl_enc, val_true_log)],
    callbacks=[lgb.early_stopping(300, verbose=False), lgb.log_evaluation(500)]
)
lgb_log_best_iters = lgb_log_model.best_iteration_

pred_log_val    = lgb_log_model.predict(X_vl_enc)
pred_ratio_log  = np.clip(np.exp(np.clip(pred_log_val, LOG_LOW, LOG_HIGH)), RATIO_LOW, RATIO_HIGH)
pred_lgblog_val = np.clip(y_val_lag1 * pred_ratio_log, 1, None)
print(f'LGB-log MAPE: {mape(y_val_true, pred_lgblog_val):.4f}% -> {score_to_points(mape(y_val_true, pred_lgblog_val))} pts')

  0%|          | 0/40 [00:00<?, ?it/s]

LGB-log best MAPE: 4.1556 %
[500]	valid_0's l1: 0.0356804
[1000]	valid_0's l1: 0.0351843
[1500]	valid_0's l1: 0.0352096
LGB-log MAPE: 4.1535% -> 91.87 pts


In [21]:
hist_cv     = df[df['month'] < val_month_cv]
last_rto_cv = hist_cv.groupby('new_id')['rto'].last()
roll3_cv    = hist_cv.groupby('new_id')['rto'].apply(lambda x: x.tail(3).mean())
roll6_cv    = hist_cv.groupby('new_id')['rto'].apply(lambda x: x.tail(6).mean())

seas_df     = df.copy()
seas_df['prev_rto'] = seas_df.groupby('new_id')['rto'].shift(1)
seas_df['ratio']    = seas_df['rto'] / seas_df['prev_rto'].replace(0, np.nan)
march_ratios        = seas_df[seas_df['month'].dt.month == 3]['ratio'].dropna()
MARCH_MED_RATIO     = march_ratios.median()
print(f'Historical March/Feb median ratio: {MARCH_MED_RATIO:.4f}')

lag1_s_val  = pd.Series(y_val_lag1, index=val_df.index)
p_last_val  = np.clip(val_df['new_id'].map(last_rto_cv).fillna(lag1_s_val).values.astype(float), 1, None)
p_roll3_val = np.clip(val_df['new_id'].map(roll3_cv).fillna(lag1_s_val).values.astype(float), 1, None)
p_roll6_val = np.clip(val_df['new_id'].map(roll6_cv).fillna(lag1_s_val).values.astype(float), 1, None)
p_sea_val   = np.clip(p_last_val * MARCH_MED_RATIO, 1, None)

print(f'Last MAPE    : {mape(y_val_true, p_last_val):.4f}%')
print(f'Roll3 MAPE   : {mape(y_val_true, p_roll3_val):.4f}%')
print(f'Roll6 MAPE   : {mape(y_val_true, p_roll6_val):.4f}%')
print(f'Seasonal MAPE: {mape(y_val_true, p_sea_val):.4f}%')

Historical March/Feb median ratio: 1.1192
Last MAPE    : 5.3429%
Roll3 MAPE   : 9.6226%
Roll6 MAPE   : 9.0517%
Seasonal MAPE: 16.3297%


In [22]:
preds_val  = [pred_cb_val, pred_lgb_val, pred_xgb_val, pred_lgblog_val,
              p_last_val, p_roll3_val, p_roll6_val, p_sea_val]
names_list = ['cb', 'lgb', 'xgb', 'lgb_log', 'last', 'roll3', 'roll6', 'seasonal']

print('Individual val MAPEs:')
for n, p in zip(names_list, preds_val):
    print(f'  {n}: {mape(y_val_true, p):.4f}%')

Individual val MAPEs:
  cb: 5.7090%
  lgb: 4.2403%
  xgb: 4.4905%
  lgb_log: 4.1535%
  last: 5.3429%
  roll3: 9.6226%
  roll6: 9.0517%
  seasonal: 16.3297%


In [23]:
P_val = np.stack(preds_val, axis=1)

def neg_mape_w(w):
    w = np.clip(w, 0, None)
    s = w.sum()
    if s == 0:
        return 1e9
    w = w / s
    return mape(y_val_true, P_val @ w)

best_res = None
for _ in range(100):
    w0  = np.random.dirichlet(np.ones(len(preds_val)))
    res = minimize(
        neg_mape_w, w0,
        method='SLSQP',
        bounds=[(0, 1)] * len(preds_val),
        constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        options={'maxiter': 1000, 'ftol': 1e-10}
    )
    if best_res is None or res.fun < best_res.fun:
        best_res = res

opt_w = np.clip(best_res.x, 0, None)
opt_w = opt_w / opt_w.sum()
pred_opt_val = P_val @ opt_w

ens_mape = mape(y_val_true, pred_opt_val)
print(f'Optimized ensemble MAPE: {ens_mape:.4f}% -> {score_to_points(ens_mape)} pts')
for n, w in zip(names_list, opt_w):
    print(f'  {n}: {w:.4f}')

Optimized ensemble MAPE: 4.1425% -> 91.89 pts
  cb: 0.0000
  lgb: 0.2299
  xgb: 0.0000
  lgb_log: 0.7701
  last: 0.0000
  roll3: 0.0000
  roll6: 0.0000
  seasonal: 0.0000


In [24]:
store_hist_min = df.groupby('new_id')['rto'].min() * 0.4
store_hist_max = df.groupby('new_id')['rto'].max() * 2.0

pred_clipped = pred_opt_val.copy()
for i, sid in enumerate(val_df['new_id'].values):
    lo = store_hist_min.get(sid, 1)
    hi = store_hist_max.get(sid, np.inf)
    pred_clipped[i] = np.clip(pred_opt_val[i], lo, hi)

clip_mape = mape(y_val_true, pred_clipped)
print(f'After postprocess MAPE: {clip_mape:.4f}% -> {score_to_points(clip_mape)} pts')

USE_POSTPROCESS = clip_mape < ens_mape
print('Use postprocessing:', USE_POSTPROCESS)

After postprocess MAPE: 4.1425% -> 91.89 pts
Use postprocessing: False


In [25]:
full_mask = df_feat['target_ratio'].notna() & df_feat['lag_1'].notna()
full_df   = df_feat[full_mask].copy()
full_df['target_ratio']     = full_df['target_ratio'].clip(RATIO_LOW, RATIO_HIGH)
full_df['log_target_ratio'] = np.log(full_df['target_ratio'])

X_full_cat = full_df[ALL_CB_FEATS].copy()
for c in CAT_COLS:
    X_full_cat[c] = X_full_cat[c].astype(str).fillna('NA')
y_full_ratio = full_df['target_ratio'].copy()
y_full_log   = full_df['log_target_ratio'].copy()
X_full_enc   = full_df[ALL_ENC_FEATS].copy()

cb_full = CatBoostRegressor(
    iterations=int(cb_best_iters * 1.1) + 1,
    depth=CB_DEPTH, learning_rate=0.01, l2_leaf_reg=3,
    min_data_in_leaf=10, subsample=0.8, colsample_bylevel=0.8,
    loss_function='MAPE', random_seed=SEED,
    task_type=CB_TASK, verbose=500, thread_count=-1
)
cb_full.fit(Pool(X_full_cat, y_full_ratio, cat_features=CAT_COLS))
print('CatBoost full trained')

lgb_full = lgb.LGBMRegressor(
    objective='mape', metric='mape',
    n_estimators=int(lgb_best_iters * 1.1) + 1,
    learning_rate=best_lgb_params['lr'],
    num_leaves=best_lgb_params['num_leaves'],
    max_depth=best_lgb_params['max_depth'],
    min_child_samples=best_lgb_params['min_child_samples'],
    subsample=best_lgb_params['subsample'],
    colsample_bytree=best_lgb_params['colsample_bytree'],
    reg_alpha=best_lgb_params['reg_alpha'],
    reg_lambda=best_lgb_params['reg_lambda'],
    min_split_gain=best_lgb_params['min_split_gain'],
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_full.fit(X_full_enc, y_full_ratio)
print('LightGBM full trained')

xgb_full = xgb.XGBRegressor(
    objective='reg:absoluteerror',
    n_estimators=int(xgb_best_iters * 1.1) + 1,
    learning_rate=best_xgb_params['lr'],
    max_depth=best_xgb_params['max_depth'],
    min_child_weight=best_xgb_params['min_child_weight'],
    subsample=best_xgb_params['subsample'],
    colsample_bytree=best_xgb_params['colsample_bytree'],
    reg_alpha=best_xgb_params['reg_alpha'],
    reg_lambda=best_xgb_params['reg_lambda'],
    gamma=best_xgb_params['gamma'],
    random_state=SEED, n_jobs=-1, tree_method='hist'
)
xgb_full.fit(X_full_enc, y_full_ratio, verbose=500)
print('XGBoost full trained')

lgb_log_full = lgb.LGBMRegressor(
    objective='regression_l1', metric='mae',
    n_estimators=int(lgb_log_best_iters * 1.1) + 1,
    learning_rate=best_lgb_log_params['lr'],
    num_leaves=best_lgb_log_params['num_leaves'],
    max_depth=best_lgb_log_params['max_depth'],
    min_child_samples=best_lgb_log_params['min_child_samples'],
    subsample=best_lgb_log_params['subsample'],
    colsample_bytree=best_lgb_log_params['colsample_bytree'],
    reg_alpha=best_lgb_log_params['reg_alpha'],
    reg_lambda=best_lgb_log_params['reg_lambda'],
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_log_full.fit(X_full_enc, y_full_log)
print('LGB-log full trained')

0:	learn: 0.0717061	total: 273ms	remaining: 0us
CatBoost full trained
LightGBM full trained
XGBoost full trained
LGB-log full trained


In [26]:
all_ids    = np.sort(df['new_id'].unique())
print('Building test features for', len(all_ids), 'stores...')

lag_maps = {}
for lag in [1, 2, 3, 4, 5, 6, 9, 12]:
    lag_maps[lag] = (
        df.sort_values(['new_id', 'month'])
          .groupby('new_id')['rto']
          .apply(lambda x: x.iloc[-lag] if len(x) >= lag else np.nan)
    )

roll_maps = {}
for w in [2, 3, 6, 9, 12]:
    roll_maps[('mean', w)] = df.groupby('new_id')['rto'].apply(lambda x, w=w: x.tail(w).mean())
    roll_maps[('std',  w)] = df.groupby('new_id')['rto'].apply(lambda x, w=w: x.tail(w).std())
    roll_maps[('min',  w)] = df.groupby('new_id')['rto'].apply(lambda x, w=w: x.tail(w).min())
    roll_maps[('max',  w)] = df.groupby('new_id')['rto'].apply(lambda x, w=w: x.tail(w).max())
    roll_maps[('med',  w)] = df.groupby('new_id')['rto'].apply(lambda x, w=w: x.tail(w).median())

latest_rows   = df.sort_values(['new_id', 'month']).groupby('new_id').last()
store_medians = df.groupby('new_id')['rto'].median()

records = []
for sid in all_ids:
    row    = {'new_id': sid}
    static = latest_rows.loc[sid] if sid in latest_rows.index else None

    for c in CAT_COLS:
        row[c] = str(static[c]) if static is not None and not pd.isna(static[c]) else 'NA'

    for c in NUM_STATIC_COLS:
        row[c] = float(static[c]) if static is not None and not pd.isna(static[c]) else np.nan

    for lag in [1, 2, 3, 4, 5, 6, 9, 12]:
        row[f'lag_{lag}'] = lag_maps[lag].get(sid, np.nan)

    for w in [2, 3, 6, 9, 12]:
        for s in ['mean', 'std', 'min', 'max', 'med']:
            row[f'roll_{s}_{w}'] = roll_maps[(s, w)].get(sid, np.nan)

    l1  = row.get('lag_1',  np.nan)
    l2  = row.get('lag_2',  np.nan)
    l3  = row.get('lag_3',  np.nan)
    l6  = row.get('lag_6',  np.nan)
    l12 = row.get('lag_12', np.nan)
    rm3 = row.get('roll_mean_3',  np.nan)
    rm6 = row.get('roll_mean_6',  np.nan)
    rm12= row.get('roll_mean_12', np.nan)
    rs3 = row.get('roll_std_3',   np.nan)
    rs6 = row.get('roll_std_6',   np.nan)

    def safe_div(a, b):
        return a / b if (not np.isnan(b) and b != 0 and not np.isnan(a)) else np.nan
    def safe_diff(a, b):
        return a - b if (not np.isnan(a) and not np.isnan(b)) else np.nan

    row['ratio_1_2']  = safe_div(l1, l2)
    row['ratio_2_3']  = safe_div(l2, l3)
    row['ratio_1_3']  = safe_div(l1, l3)
    row['ratio_1_6']  = safe_div(l1, l6)
    row['ratio_1_12'] = safe_div(l1, l12)
    row['ratio_3_12'] = safe_div(l3, l12)
    row['diff_1_2']   = safe_diff(l1, l2)
    row['diff_1_3']   = safe_diff(l1, l3)
    row['diff_1_12']  = safe_diff(l1, l12)
    row['cv_3']       = safe_div(rs3, rm3)
    row['cv_6']       = safe_div(rs6, rm6)
    row['trend_3_6']  = safe_div(rm3, rm6)
    row['trend_6_12'] = safe_div(rm6, rm12)
    row['trend_3_12'] = safe_div(rm3, rm12)
    row['range_6']    = safe_diff(row.get('roll_max_6', np.nan), row.get('roll_min_6', np.nan))
    row['range_12']   = safe_diff(row.get('roll_max_12', np.nan), row.get('roll_min_12', np.nan))
    row['log_lag_1']  = np.log1p(l1)  if not np.isnan(l1)  else np.nan
    row['log_lag_3']  = np.log1p(l3)  if not np.isnan(l3)  else np.nan
    row['log_lag_12'] = np.log1p(l12) if not np.isnan(l12) else np.nan

    row['month_num']  = 3
    row['year']       = 2025
    row['month_sin']  = float(np.sin(2 * np.pi * 3 / 12))
    row['month_cos']  = float(np.cos(2 * np.pi * 3 / 12))
    row['quarter']    = 1
    row['is_q1']      = 1
    row['is_march']   = 1

    sm = store_medians.get(sid, np.nan)
    row['store_size']     = sm
    row['log_store_size'] = np.log1p(sm) if not np.isnan(sm) else np.nan
    row['seasonal_idx']   = np.nan

    ft  = row.get('foot_traffic', np.nan)
    ct  = row.get('car_traffic',  np.nan)
    avg_p = row.get('avg_promo_items',   np.nan)
    avg_i = row.get('avg_items_in_check',np.nan)
    wh    = row.get('working_hours',     np.nan)
    nc    = row.get('num_cashiers',      np.nan)
    pop   = row.get('population',        np.nan)
    hh    = row.get('households',        np.nan)
    g500  = row.get('grocery_500m',      np.nan)
    p500  = row.get('pyaterochka_500m',  np.nan)
    st300 = row.get('stops_300m',        np.nan)
    sc300 = row.get('schools_300m',      np.nan)
    md300 = row.get('medical_300m',      np.nan)

    row['promo_x_traffic']    = safe_div(avg_p * ft if not np.isnan(avg_p) and not np.isnan(ft) else np.nan, 1) if not np.isnan(avg_p) and not np.isnan(ft) else np.nan
    row['promo_x_traffic']    = avg_p * ft if not (np.isnan(avg_p) or np.isnan(ft)) else np.nan
    row['items_x_hours']      = avg_i * wh if not (np.isnan(avg_i) or np.isnan(wh)) else np.nan
    row['competition_idx']    = g500 + p500 if not (np.isnan(g500) or np.isnan(p500)) else np.nan
    row['infra_idx']          = st300 + sc300 + md300 if not any(np.isnan(x) for x in [st300, sc300, md300]) else np.nan
    row['total_traffic']      = ft + ct if not (np.isnan(ft) or np.isnan(ct)) else np.nan
    row['households_per_cash']= safe_div(hh, nc)
    row['pop_per_cash']       = safe_div(pop, nc)

    records.append(row)

test_raw = pd.DataFrame(records)
print('Test rows built:', len(test_raw))

Building test features for 18657 stores...
Test rows built: 18657


In [28]:
X_test_cat = test_raw[ALL_CB_FEATS].copy()
for c in CAT_COLS:
    X_test_cat[c] = X_test_cat[c].astype(str).fillna('NA')

X_test_enc = test_raw[ALL_NUM_FEATS].copy()

for c in CAT_COLS:
    le = le_dict[c]
    col_raw = test_raw[c].astype(str).fillna('NA')
    known = set(le.classes_)
    col_raw = col_raw.map(lambda v: v if v in known else le.classes_[0])
    X_test_enc[c + '_enc'] = le.transform(col_raw)

X_test_enc = X_test_enc[ALL_ENC_FEATS]

lag1_test = test_raw['lag_1'].values.astype(float)

pr_cb  = np.clip(cb_full.predict(Pool(X_test_cat, cat_features=CAT_COLS)), RATIO_LOW, RATIO_HIGH)
pr_lgb = np.clip(lgb_full.predict(X_test_enc),    RATIO_LOW, RATIO_HIGH)
pr_xgb = np.clip(xgb_full.predict(X_test_enc),    RATIO_LOW, RATIO_HIGH)
pr_log = np.clip(np.exp(np.clip(lgb_log_full.predict(X_test_enc), LOG_LOW, LOG_HIGH)), RATIO_LOW, RATIO_HIGH)

p_cb_t  = np.clip(lag1_test * pr_cb,  1, None)
p_lgb_t = np.clip(lag1_test * pr_lgb, 1, None)
p_xgb_t = np.clip(lag1_test * pr_xgb, 1, None)
p_log_t = np.clip(lag1_test * pr_log,  1, None)

last_all  = df.groupby('new_id')['rto'].last()
roll3_all = df.groupby('new_id')['rto'].apply(lambda x: x.tail(3).mean())
roll6_all = df.groupby('new_id')['rto'].apply(lambda x: x.tail(6).mean())

lag1_s_t  = pd.Series(lag1_test, index=test_raw.index)
p_last_t  = np.clip(test_raw['new_id'].map(last_all).fillna(lag1_s_t).values.astype(float), 1, None)
p_roll3_t = np.clip(test_raw['new_id'].map(roll3_all).fillna(lag1_s_t).values.astype(float), 1, None)
p_roll6_t = np.clip(test_raw['new_id'].map(roll6_all).fillna(lag1_s_t).values.astype(float), 1, None)
p_sea_t   = np.clip(p_last_t * MARCH_MED_RATIO, 1, None)

for arr in [p_last_t, p_roll3_t, p_roll6_t, p_sea_t]:
    nan_idx = np.isnan(arr)
    if nan_idx.sum() > 0:
        arr[nan_idx] = np.nanmedian(arr[~nan_idx])

P_test     = np.stack([p_cb_t, p_lgb_t, p_xgb_t, p_log_t,
                       p_last_t, p_roll3_t, p_roll6_t, p_sea_t], axis=1)
pred_final = P_test @ opt_w
pred_final = np.clip(pred_final, 1, None)

if USE_POSTPROCESS:
    store_hist_min = df.groupby('new_id')['rto'].min() * 0.4
    store_hist_max = df.groupby('new_id')['rto'].max() * 2.0
    for i, sid in enumerate(test_raw['new_id'].values):
        lo = store_hist_min.get(sid, 1)
        hi = store_hist_max.get(sid, np.inf)
        pred_final[i] = np.clip(pred_final[i], lo, hi)
    print('Postprocessing applied')

nan_mask = np.isnan(pred_final)
if nan_mask.sum() > 0:
    pred_final[nan_mask] = np.nanmedian(pred_final)
    print(f'Filled {nan_mask.sum()} NaNs with median')

pred_final = np.clip(pred_final, 1, None)
print('Pred stats:')
print(pd.Series(pred_final).describe())

Pred stats:
count    1.865700e+04
mean     1.004928e+08
std      5.466331e+07
min      1.486269e+07
25%      6.409724e+07
50%      8.529073e+07
75%      1.188205e+08
max      5.339055e+08
dtype: float64


In [29]:
submission = pd.DataFrame({'new_id': test_raw['new_id'].values, 'rto': pred_final})

n = len(submission)
print(f'Rows: {n} (expected 18657)')

if n < 18657:
    med = submission['rto'].median()
    extra = pd.DataFrame({'new_id': [f'MISSING_{i}' for i in range(18657 - n)], 'rto': med})
    submission = pd.concat([submission, extra], ignore_index=True)
    print(f'WARNING: added {18657 - n} placeholder rows')

if n > 18657:
    submission = submission.sort_values('new_id').head(18657).reset_index(drop=True)
    print(f'WARNING: trimmed to 18657 rows')

assert len(submission) == 18657,                             f'Wrong rows: {len(submission)}'
assert submission['new_id'].nunique() == len(submission),    'Duplicate new_id!'
assert submission['rto'].isna().sum() == 0,                  'NaN in rto!'
assert (submission['rto'] < 0).sum() == 0,                   'Negative rto!'

submission.to_csv('test.csv', index=False)
print('All checks passed!')
print('test.csv saved with', len(submission), 'rows')
print(submission.head())

Rows: 18657 (expected 18657)
All checks passed!
test.csv saved with 18657 rows
   new_id           rto
0       0  9.055916e+07
1       1  4.330651e+07
2       2  8.769618e+07
3       3  6.572099e+07
4       4  8.044236e+07
